# 01 · Clock and acquisition
**CVES manuscript repair series**

| | |
|---|---|
| **Steps** | 0, 1, 2 (clock gate, regime tables are completed in notebook 04) |
| **Needs** | Notebook 00 done; CVES dataset |
| **Produces** | `0_known_answers`, `1_acquisition`, `step1.pkl` (private), clock figure |
| **Runtime** | about 5–10 minutes |
| **Done when** | at least 90% of sit-stand-balance records pass the plausibility gate at 1000 Hz, and no time-based number in your analysis uses 500 Hz for this directory |

**How to use.** Run the cells top to bottom. Each notebook has the same five parts: **0 Setup** → **1 Run the kit step** → **2 Compare with the reference run** → **3 Tables and figures** → **4 Where each number goes**.
All numbers are *exploratory* until regenerated from a frozen commit (notebook 05). Everything printed here is an aggregate; per-participant files stay in the private folder.

In [ ]:
#@title 0. Setup (run in every session) { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
!pip -q install wfdb
import sys; sys.path.insert(0, '/content/drive/MyDrive/CVES Fresh/CVES Repair/kit')
import helpers as H
cfg = H.setup()

## 1 · Run the kit steps
**Why.** The `sit-stand-balance` headers say 500 Hz, but the recordings ran at 1000 Hz. The kit uses `CLOCK = {'sit-stand-balance': 1000, 'head-up-tilt': 500, 'sit-stand': 500}` in `core.py`. This step measures heart rate, breathing, timing, and group links on the corrected clock.

In [ ]:
#@title 1a. Known-answer tests (must all pass) { display-mode: "form" }
H.run_step(cfg, '0_known_answers', 'known_answers.py')

In [ ]:
#@title 1b. Acquisition audit: clock, mains line, ECG anchoring, regime by gain/timing { display-mode: "form" }
# Writes private/step1.pkl (per-record rows; stays private) and reports/1_acquisition.txt (aggregates).
H.run_step(cfg, '1_acquisition', 'step1_acquisition.py', cfg.data, cfg.work / 'step1.pkl')

In [ ]:
#@title 2. Compare with the reference run { display-mode: "form" }
H.compare(cfg, '0_known_answers')
H.compare(cfg, '1_acquisition')
# Rule of thumb: counts exact, AUC +/-0.01, phase and timing +/-1 degree.
# Differences beyond that are listed above. Library versions can cause small ones; report the rest as implementation sensitivity.

## 3 · Tables and figures

In [ ]:
#@title 3a. Plausibility gate (the "Done when" test) { display-mode: "form" }
ok = H.clock_gate(cfg.work / 'step1.pkl')
print('GATE PASSED. Continue.' if ok else 'GATE FAILED. Stop. Do not patch individual records.')
print()
print(H.grep(H.read_report(cfg, '1_acquisition'), r'as labelled|failing records|same person tilt/seated'))

In [ ]:
#@title 3b. Figure: heart rate and breathing as labelled, as corrected, and in the supine session { display-mode: "form" }
fig = H.clock_figure(cfg.work / 'step1.pkl', cfg.figs / 'fig_clock_heart_rate.png')
print('Saved', cfg.figs / 'fig_clock_heart_rate.png')

In [ ]:
#@title 3c. Worked example check: 35 per minute at 500 Hz becomes 70 per minute at 1000 Hz { display-mode: "form" }
import pandas as pd
d = pd.read_pickle(cfg.work / 'step1.pkl'); s = d[d.proto == 'sit-stand-balance']
print('Median heart rate, as labelled (500 Hz):   %.1f per minute' % (s.hr.median() / 2))
print('Median heart rate, corrected (1000 Hz):    %.1f per minute' % s.hr.median())
print('Median breathing, as labelled / corrected: %.1f / %.1f per minute' % (s.br.median() / 2, s.br.median()))
print('Median duration, as labelled / corrected:  %.0f s / %.0f s' % (s.dur.median() * 2, s.dur.median()))
print('Reference run: 34.7 -> 69.5 per minute; 7.9 -> 15.8 breaths; 2179 -> 1090 s')

In [ ]:
#@title (Optional) Raw lab-file census: slow, needs the 77 GB raw-data.tar.gz { display-mode: "form" }
RAW_ARCHIVE = "" #@param {type:"string"}
# Leave blank unless the raw archive is mounted. Reads every original WinDAQ file once (hours on a laptop).
if RAW_ARCHIVE:
    H.run_step(cfg, 'raw_census', 'raw_clock_census.py', RAW_ARCHIVE, cfg.work / 'step1.pkl')
else:
    print('Skipped. Quote the reference census (reference/raw_census.txt) and say so in the paper.')

## 4 · Where each number goes
| Manuscript | Take from |
|---|---|
| **S1** (clock paragraph) | `1_acquisition`: records passing the gate, `as labelled at 500 Hz` line; the mains-line count (46 of 64) is in `7_robustness` (notebook 04) |
| **R1** (heart rate and breathing) | `1_acquisition`; same-person tilt/seated ratios |
| **New figure** (heart rate as labelled, corrected, supine) | `figures/fig_clock_heart_rate.png` |
| **Do not cite** | the gain-based regime table in `1_acquisition` as the paper's definition: it is a comparator. The paper's definition (left span) is in notebook 04 |